# TrajAI Model Training - Full State Prediction

This notebook trains the LSTM model to predict the **entire vehicle state (12 features)**, not just position.

**Features Predicted:**
1. **Position**: `Local_X`, `Local_Y`
2. **Dynamics**: `v_Vel` (Velocity), `v_Acc` (Acceleration)
3. **Context**: `Space_Headway`, Lane Position (`dis_cen`, `dis_l`, `dis_r`, `dis_f`), Lane Indicators (`i_l`, `i_r`, `i_f`)

**Output Files:**
- `trajectory_lstm_best.pth`
- `scalers.pkl`

In [ ]:
import os
import glob
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything()

## Configuration

In [ ]:
class Config:
    WINDOW_SIZE = 20
    PREDICTION_HORIZON = 3  # Predict 3 steps ahead (autoregressive training)
    INPUT_SIZE = 12
    OUTPUT_SIZE = 12        # NOW PREDICTING ALL 12 FEATURES
    HIDDEN_SIZE = 128      
    NUM_LAYERS = 2
    DROPOUT = 0.2
    BATCH_SIZE = 64
    EPOCHS = 50            
    LEARNING_RATE = 1e-3
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    FEATURE_COLUMNS = [
        "Local_X", "Local_Y", "v_Vel", "v_Acc", "Space_Headway", "dis_cen",
        "i_l", "i_r", "i_f", "dis_l", "dis_r", "dis_f"
    ]
    # Target is now the same as features
    TARGET_COLUMNS = FEATURE_COLUMNS 

print(f"Using device: {Config.DEVICE}")

## Model Definition

In [ ]:
class TrajectoryLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, output_size=12, num_layers=2, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, output_size)  # Outputs all features

    def forward(self, x):
        outputs, _ = self.lstm(x)
        last_hidden = outputs[:, -1, :]
        return self.head(last_hidden)

## Data Loading

In [ ]:
def load_and_prepare_data(raw_data_dir):
    """
    Load CSVs and create sequences WITHOUT mixing different vehicle trajectories.
    """
    csv_files = sorted(glob.glob(os.path.join(raw_data_dir, "*.csv")))
    all_sequences = []
    all_targets = []
    
    MAX_FILES = 200 # Limit for speed if needed, remove for full training
    
    for csv_path in tqdm(csv_files[:MAX_FILES], desc="Processing CSVs"):
        try:
            df = pd.read_csv(csv_path, usecols=Config.FEATURE_COLUMNS + ['Vehicle_ID'])
        except ValueError:
            # Fallback if vehicle ID missing
            df = pd.read_csv(csv_path)
            if not set(Config.FEATURE_COLUMNS).issubset(df.columns):
                continue
            df = df[Config.FEATURE_COLUMNS].copy()
            df['Vehicle_ID'] = 0
        
        df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        
        for vid, vdf in df.groupby('Vehicle_ID'):
            vdf = vdf.sort_index()
            features = vdf[Config.FEATURE_COLUMNS].values.astype(np.float32)
            
            if len(features) < Config.WINDOW_SIZE + 1:
                continue
            
            # Create sequences
            # We predict the NEXT step (t+1)
            for i in range(len(features) - Config.WINDOW_SIZE):
                window = features[i : i + Config.WINDOW_SIZE]
                target = features[i + Config.WINDOW_SIZE] # Predict full state at t+1
                
                all_sequences.append(window)
                all_targets.append(target)
    
    return np.array(all_sequences), np.array(all_targets)

# Check for Kaggle path or local
DATA_DIR = "/kaggle/input/car-data/car_data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "data/raw"

print(f"Loading from: {DATA_DIR}")
X_raw, y_raw = load_and_prepare_data(DATA_DIR)
print(f"Total sequences: {len(X_raw)}")

## Scaling

In [ ]:
# Reshape X to 2D for scaler fitting
X_2d = X_raw.reshape(-1, Config.INPUT_SIZE)

print("Fitting scaler on all features...")
scaler = MinMaxScaler()
scaler.fit(X_2d)

joblib.dump(scaler, "scalers.pkl")
print("Scaler saved.")

X_scaled = scaler.transform(X_2d).reshape(X_raw.shape)
y_scaled = scaler.transform(y_raw) # Directly transform full target vectors

print(f"X_scaled shape: {X_scaled.shape}")
print(f"y_scaled shape: {y_scaled.shape}")

## Dataset

In [ ]:
class TrajectoryDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.from_numpy(features).float()
        self.targets = torch.from_numpy(targets).float()
    def __len__(self): return len(self.features)
    def __getitem__(self, idx): return self.features[idx], self.targets[idx]

split_idx = int(len(X_scaled) * 0.8)
train_dataset = TrajectoryDataset(X_scaled[:split_idx], y_scaled[:split_idx])
val_dataset = TrajectoryDataset(X_scaled[split_idx:], y_scaled[split_idx:])

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)

## Training

In [ ]:
model = TrajectoryLSTM(
    input_size=Config.INPUT_SIZE,
    hidden_size=Config.HIDDEN_SIZE,
    output_size=Config.OUTPUT_SIZE,
    num_layers=Config.NUM_LAYERS,
    dropout=Config.DROPOUT
).to(Config.DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=Config.LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': []}

print("Starting training (12-feature prediction)...")

for epoch in range(Config.EPOCHS):
    # Train
    model.train()
    train_losses = []
    for x, y in train_loader:
        x, y = x.to(Config.DEVICE), y.to(Config.DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    
    # Val
    model.eval()
    val_losses = []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(Config.DEVICE), y.to(Config.DEVICE)
            out = model(x)
            loss = criterion(out, y)
            val_losses.append(loss.item())
            
    avg_train = np.mean(train_losses)
    avg_val = np.mean(val_losses)
    scheduler.step(avg_val)
    
    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    
    print(f"Epoch {epoch+1} | Train: {avg_train:.6f} | Val: {avg_val:.6f}")
    
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "trajectory_lstm_best.pth")
        print("  >>> Model saved!")

## Validation Check

In [ ]:
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Val')
plt.legend()
plt.title("Loss Curve")
plt.show()

In [ ]:
# Sanity Check: Predict velocity and headway
model.load_state_dict(torch.load("trajectory_lstm_best.pth", weights_only=True))
model.eval()

sample_x, sample_y = val_dataset[10]
with torch.no_grad():
    pred = model(sample_x.unsqueeze(0).to(Config.DEVICE))
    pred = pred.cpu().numpy()[0]

# Inverse transform to see real values
real_pred = scaler.inverse_transform([pred])[0]
real_target = scaler.inverse_transform([sample_y.numpy()])[0]

cols = Config.FEATURE_COLUMNS
df_res = pd.DataFrame({'Predicted': real_pred, 'Actual': real_target}, index=cols)
print(df_res)

print("\nErrors:")
print(np.abs(real_pred - real_target))